In [1]:
#packages
import requests
import pandas as pd
import json
from glob import glob
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from censusdis import data
from censusdis.datasets import ACS5
from scipy.spatial import cKDTree
from shapely.geometry import Point
import geopandas as gpd

# Socioeconomic dataset
One row per tract.

Returns: tract_socioeconomic


In [2]:
vars = [
    # Population
    "B01003_001E",    # total population

    # Income & Poverty
    "B19013_001E",    # median household income
    "C17002_002E",    # below 50% poverty (numerator for severe poverty)

    # Housing & Vehicles (numerators + denominators)
    "B25044_001E",    # households for vehicle availability (denominator)
    "B25044_003E",    # households with NO vehicle (numerator)

    # Race / Ethnicity (counts)
    "B02001_002E", "B02001_003E", "B02001_004E",
    "B02001_005E", "B02001_006E", "B02001_007E", "B02001_008E",

    # Housing Cost Burden (rent % of income)
    "B25070_001E", "B25070_007E", "B25070_008E", "B25070_009E", "B25070_010E",

    # Education
    "B15003_022E",   # bachelor's degree (if we want bachelors+, consider adding 023-025)

    # Disability (numerator)
    "B18135_002E",   # population WITH a disability (numerator)
    # NOTE: also add B18135_001E (denominator) below

    # Internet Access (numerator)
    "B28002_002E",   # households with broadband subscription (numerator)
    # NOTE: also add B28002_001E (denominator) below

    # Age 65+ (WE ADD the full set of bins to sum up correctly)
    # Male 65-66 .. 85+
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E","B01001_024E","B01001_025E",
    # Female 65-66 .. 85+
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E","B01001_048E","B01001_049E",

    # Housing Tenure (optional)
    "B25003_003E",   # renter-occupied units (numerator)
    "B25003_001E",   # total housing units for tenure table (denominator)
    
    # Disability and internet DENOMINATORS (needed to compute rates correctly)
    "B18135_001E",   # total civilian noninstitutionalized population (disability denominator)
    "B28002_001E"    # total households for internet table (internet denominator)
]

#Pulling this data for all census tracts in NC

In [3]:
dataset = "acs/acs5"
vintage = 2022        #2024 comes out on Dec 11
state_fips = "37"     # North Carolina as string

df = data.download(
    dataset=dataset,
    vintage=vintage,
    download_variables=vars,
    # Geographic filters (must be strings)
    state=state_fips,
    county="*",        # all counties
    tract="*"          # all tracts within each county
)

#derived indicators

#Getting percentages, replace any with NA denominator with 0
# safe pct helper (keeps your original fill-0 behavior)
def pct(num, den):
    den_safe = den.replace(0, np.nan)
    result = num / den_safe
    return result.fillna(0)

# Severe poverty
df["pct_below_50pct_poverty"] = pct(df["C17002_002E"], df["B01003_001E"])

# No vehicle
df["pct_no_vehicle"] = pct(df["B25044_003E"], df["B25044_001E"])

# Renters cost burden (sum of 30%+ buckets / total renters)
df["pct_renters_cost_burdened"] = pct(
    df["B25070_007E"] + df["B25070_008E"] + df["B25070_009E"] + df["B25070_010E"],
    df["B25070_001E"]
)

# Percent 65+ : sum the male+female 65+ bins, divide by total population (B01003_001E)
age_cols_65 = [
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E","B01001_024E","B01001_025E",
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E","B01001_048E","B01001_049E"
]
df["population_over65"] = df[age_cols_65].sum(axis=1)
df["pct_over65"] = pct(df["population_over65"], df["B01003_001E"])

# Disability: use ACS numerator (people with disability) / ACS denominator (civilian noninstitutionalized)
# numerator: B18135_002E ; denominator: B18135_001E
df["pct_with_disability"] = pct(df["B18135_002E"], df["B18135_001E"])

# Optional broadband if we want (uncomment if downloaded B28002_001E)
# df["pct_broadband"] = pct(df["B28002_002E"], df["B28002_001E"])

# Percent nonwhite: 1 - (white / total_race_counts)
race_cols = ["B02001_002E","B02001_003E","B02001_004E","B02001_005E","B02001_006E","B02001_007E","B02001_008E"]
race_total = df[race_cols].sum(axis=1)
df["pct_nonwhite"] = 1 - pct(df["B02001_002E"], race_total)
#df["pct_renters"] = pct(df["B25003_003E"], df["B25003_001E"])

#save dataframe
#df.to_csv("./data_processed/nc_census_tracts_acs5.csv", index=False)

'''
#downloading tract boundaries for mapping
df_geo = data.download(dataset=dataset, vintage=year,
                       variables=vars, geography="tract",
                       state=state, with_geometry=True)
df_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")
'''

'\n#downloading tract boundaries for mapping\ndf_geo = data.download(dataset=dataset, vintage=year,\n                       variables=vars, geography="tract",\n                       state=state, with_geometry=True)\ndf_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")\n'

In [5]:
duplicate_counts = df['TRACT'].duplicated().sum()
print("Number of duplicate TRACT values:", duplicate_counts)

#drop duplicates and keep first occurrence
df = df.drop_duplicates(subset='TRACT', keep='first')

Number of duplicate TRACT values: 896


In [6]:
rename_dict = {
    # Population
    "B01003_001E": "total_population",

    # Income & Poverty
    "B19013_001E": "median_household_income",
    "C17002_002E": "population_below_50pct_poverty",

    # Housing & Vehicles
    "B25044_001E": "total_households_vehicle_data",
    "B25044_003E": "households_no_vehicle",

    # Race / Ethnicity counts
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B02001_004E": "american_indian_alaska_native",
    "B02001_005E": "asian_alone",
    "B02001_006E": "native_hawaiian_pacific_islander",
    "B02001_007E": "some_other_race",
    "B02001_008E": "two_or_more_races",

    # Housing Cost Burden (Gross Rent as % of Income)
    "B25070_001E": "total_renters",
    "B25070_007E": "rent_30_to_34_pct_income",
    "B25070_008E": "rent_35_to_39_pct_income",
    "B25070_009E": "rent_40_to_49_pct_income",
    "B25070_010E": "rent_50_plus_pct_income",

    # Education (bachelor’s only, unless adding more later)
    "B15003_022E": "bachelors_degree",

    # Disability (denominator + numerator)
    "B18135_001E": "population_civilian_noninstitutionalized",
    "B18135_002E": "population_with_disability",

    # Internet Access
    "B28002_001E": "total_households_in_internet_table",
    "B28002_002E": "households_with_broadband",

    # Age 65+ bins (male + female)
    "B01001_020E": "male_65_66",
    "B01001_021E": "male_67_69",
    "B01001_022E": "male_70_74",
    "B01001_023E": "male_75_79",
    "B01001_024E": "male_80_84",
    "B01001_025E": "male_85_plus",

    "B01001_044E": "female_65_66",
    "B01001_045E": "female_67_69",
    "B01001_046E": "female_70_74",
    "B01001_047E": "female_75_79",
    "B01001_048E": "female_80_84",
    "B01001_049E": "female_85_plus",

    # Housing Tenure
    "B25003_001E": "total_housing_units",
    "B25003_003E": "renter_occupied_housing_units",
}


df = df.rename(columns=rename_dict)


In [7]:
from sklearn.preprocessing import StandardScaler

df['TRACT'] = df['TRACT'].astype(str).str.zfill(6)

#Just wanted to save the unscaled version as well
tract_socioeconomic_unscaled = df

# Scale all numeric columns EXCEPT TRACT, STATE, COUNTY
cols_to_scale = df.columns.difference(['TRACT', 'STATE', 'COUNTY'])

scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[cols_to_scale])

df_scaled = pd.DataFrame(
    scaled_values,
    columns=cols_to_scale,
    index=df.index
)

# Add back TRACT id
df_scaled['TRACT'] = df['TRACT']

# Drop STATE and COUNTY from df_scaled (now correctly)
df_scaled = df_scaled.drop(columns=['STATE', 'COUNTY'], errors='ignore')

# Final PCA-ready socioeconomic dataframe
tract_socioeconomic = df_scaled

tract_socioeconomic.head()

,american_indian_alaska_native,asian_alone,bachelors_degree,black_alone,female_65_66,female_67_69,female_70_74,female_75_79,female_80_84,female_85_plus,...,renter_occupied_housing_units,some_other_race,total_households_in_internet_table,total_households_vehicle_data,total_housing_units,total_population,total_renters,two_or_more_races,white_alone,TRACT
0,-0.168709,-0.240665,0.246170,0.241907,-0.768385,1.404298,0.687250,0.398435,0.842291,-0.026501,...,1.389508,0.235289,0.613058,0.613058,0.613058,0.229874,1.389508,-0.387310,0.236991,020100
1,-0.168709,-0.406288,-1.084951,0.494952,-0.564672,-1.145148,0.607834,-0.320363,-0.381377,-0.779679,...,0.436308,3.704468,-0.117563,-0.117563,-0.117563,0.083436,0.436308,-0.928850,-0.556554,020200
2,-0.168709,-0.446266,-1.018498,0.372231,0.997129,0.017099,-0.358392,-0.104724,1.046235,2.811056,...,0.718997,3.499917,-0.157904,-0.157904,-0.157904,-0.032494,0.718997,-0.933048,-0.577797,020301
3,-0.017690,-0.446266,-0.389279,-0.151237,-1.107907,-0.170360,-0.411336,-0.302393,-0.636308,-0.429363,...,-0.881352,2.845355,-0.598667,-0.598667,-0.598667,-0.349777,-0.881352,-0.223588,-0.665512,020302
4,-0.168709,-0.446266,-0.493111,2.695256,-0.225150,3.466351,-0.345156,-0.625853,1.173701,-0.551974,...,1.356800,1.757146,1.558831,1.558831,1.558831,1.452965,1.356800,2.866133,-0.536681,020400


In [8]:
tract_socioeconomic.dtypes

american_indian_alaska_native               float64
asian_alone                                 float64
bachelors_degree                            float64
black_alone                                 float64
female_65_66                                float64
female_67_69                                float64
female_70_74                                float64
female_75_79                                float64
female_80_84                                float64
female_85_plus                              float64
households_no_vehicle                       float64
households_with_broadband                   float64
male_65_66                                  float64
male_67_69                                  float64
male_70_74                                  float64
male_75_79                                  float64
male_80_84                                  float64
male_85_plus                                float64
median_household_income                     float64
native_hawai

# Outage-level dataset

Returns: outage_events

In [7]:
import re

outages_raw = pd.read_csv('/Users/JChuang/Documents/3S MP/MP/data_raw/outage_tracker.csv')

#Filter out North Carolina
outages_NC = outages_raw[outages_raw['state'] == "North Carolina"]
#Turn outage_start_estimate and outage_end_estimate to UTC datetime objects
outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
outages_NC['outage_end_estimate'] = pd.to_datetime(outages_NC['outage_end_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')

#Turn fix_duration_estimate into hours
def duration_to_hours(duration_str):
    if pd.isna(duration_str):
        return None
    match = re.match(r'(?:(\d+)d )?(?:(\d+)h )?(?:(\d+)m )?(?:(\d+)s)?', duration_str)
    if not match:
        return None
    days = int(match.group(1)) if match.group(1) else 0
    hours = int(match.group(2)) if match.group(2) else 0
    minutes = int(match.group(3)) if match.group(3) else 0
    seconds = int(match.group(4)) if match.group(4) else 0
    total_hours = days * 24 + hours + minutes / 60 + seconds / 3600
    return total_hours

outages_NC['fix_duration_hours'] = outages_NC['fix_duration_estimate'].apply(duration_to_hours)

outages_NC = outages_NC.sort_values("outage_start_estimate").reset_index(drop=True)


/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_16720/705568491.py:8: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_16720/705568491.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_16720/705568491.py:9: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.

In [8]:
outages_NC['block_fips'] = outages_NC['block_fips'].astype(str)

#Take out tract and county from block FIPS
outages_NC['TRACT'] = outages_NC['block_fips'].str[5:11]

outage_events = outages_NC

# Ensure outage start/end are datetimes and in UTC
outage_events['outage_start_estimate'] = pd.to_datetime(outage_events['outage_start_estimate'], utc=True)
outage_events['outage_end_estimate']   = pd.to_datetime(outage_events['outage_end_estimate'], utc=True)

# Define Helene UTC window
start_helene = pd.Timestamp("2024-09-26 00:00", tz="UTC")
end_helene   = pd.Timestamp("2024-09-30 23:59", tz="UTC")

# Flag outages that overlap the Helene window.
# Overlap condition: outage_start <= end_helene and outage_end >= start_helene
outage_events['overlaps_helene'] = (
    (outage_events['outage_start_estimate'] <= end_helene) &
    (outage_events['outage_end_estimate'] >= start_helene)
)

# Quick counts
n_overlap = outage_events['overlaps_helene'].sum()
print(f"Outages overlapping Helene: {n_overlap} / {len(outage_events)}")

outage_events = outage_events[~outage_events['overlaps_helene']].copy()



Outages overlapping Helene: 11125 / 343190


In [ ]:
outage_events.columns

Index(['outage_identifer', 'device_lat', 'device_lon', 'block_fips',
       'convex_hull', 'jurisdiction', 'origin', 'state', 'county', 'affected',
       'cause', 'outage_start_estimate', 'outage_end_estimate',
       'fix_duration_estimate', 'outage_restored', 'fix_duration_hours',
       'TRACT', 'overlaps_helene'],
      dtype='object')

# Tract-level outage dataset

Returns: tract_outages


In [10]:
#create customer minutes out
outage_events['customer_minutes_out'] = outage_events['fix_duration_hours'] * outage_events['affected'] * 60
#Add seasonal categories
outage_events['month'] = outage_events['outage_start_estimate'].dt.month
outage_events['is_summer'] = outage_events['month'].isin([6, 7, 8])
outage_events['is_winter'] = outage_events['month'].isin([12, 1, 2])


In [11]:
tract_outage_stats = outage_events.groupby("TRACT").agg(
    total_outages = ("outage_identifer", "count"),

    # Duration metrics
    mean_duration_hours = ("fix_duration_hours", "mean"),
    median_duration_hours = ("fix_duration_hours", "median"),
    max_duration_hours = ("fix_duration_hours", "max"),
    total_duration_hours = ("fix_duration_hours", "sum"),

    # Customers affected
    mean_customers_affected = ("affected", "mean"),
    max_customers_affected = ("affected", "max"),
    total_customers_affected = ("affected", "sum"),

    # CMO
    total_customer_minutes_out = ("customer_minutes_out", "sum"),

    # Seasonal
    pct_outages_in_summer = ("is_summer", "mean"),
    pct_outages_in_winter = ("is_winter", "mean"),
).reset_index()


In [12]:
tract_outages = tract_outage_stats
tract_outages.dtypes

TRACT                          object
total_outages                   int64
mean_duration_hours           float64
median_duration_hours         float64
max_duration_hours            float64
total_duration_hours          float64
mean_customers_affected       float64
max_customers_affected          int64
total_customers_affected        int64
total_customer_minutes_out    float64
pct_outages_in_summer         float64
pct_outages_in_winter         float64
dtype: object

# Tract-level weather summary dataset

One row per tract.

Returns: tract_weather


In [14]:
#create a mapping: which tract gets which weather station name
data_folder = Path('/Users/JChuang/Documents/3S MP/MP/econet_data')

#Creating an empty dictionary to hold DataFrames for each station
station_dfs = {}

#Loop through all CSV files in the folder
for file in data_folder.glob("hourly_data_*.csv"):
    #Extract station name from filename
    station_name = file.stem.split("_")[-1]

    #Read CSV into DataFrame
    df = pd.read_csv(file)

    #Add to dictionary, appending if station already exists
    if station_name in station_dfs:
        station_dfs[station_name] = pd.concat([station_dfs[station_name], df], ignore_index = True)
    else:
        station_dfs[station_name] = df


#one combined DataFrame with a new column for station name
combined_df = pd.concat(
    [df.assign(station=station) for station, df in station_dfs.items()],
    ignore_index=True
)


In [15]:
# Load NC tract shapefile from TIGER (2020 tracts for NC: state FIPS 37)
tiger_url = "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_37_tract.zip"
gdf_tracts = gpd.read_file(tiger_url)

# Keep only necessary columns
gdf_tracts = gdf_tracts[['GEOID', 'geometry']].rename(columns={'GEOID': 'TRACT'})

# Compute centroids in a projected CRS for accurate distances
# Project to Web Mercator (meters) for distance calculations
gdf_tracts_proj = gdf_tracts.to_crs(epsg=3857)

# Compute centroids in projected CRS
gdf_tracts_proj['centroid_geom'] = gdf_tracts_proj.geometry.centroid

# Make a GeoDataFrame of centroids and transform back to EPSG:4326 (lon/lat) for readability
gdf_centroids = gpd.GeoDataFrame(
    gdf_tracts_proj[['TRACT']].copy(),
    geometry=gdf_tracts_proj['centroid_geom'],
    crs="EPSG:3857"
).to_crs(epsg=4326)

# Extract lon/lat
gdf_centroids['tract_lon'] = gdf_centroids.geometry.x
gdf_centroids['tract_lat'] = gdf_centroids.geometry.y

# Prepare weather station table (unique stations with lat/lon)
# If your combined_df has different column names, change them here
weather_stations = combined_df[['location_id', 'latitude_degrees_north', 'longitude_degrees_east']].drop_duplicates().rename(
    columns={'latitude_degrees_north': 'station_lat', 'longitude_degrees_east': 'station_lon', 'location_id':'station'}
).reset_index(drop=True)

# Convert stations to GeoDataFrame and project to same projected CRS (3857) for KDTree
gdf_stations = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations['station_lon'], weather_stations['station_lat']),
    crs="EPSG:4326"
).to_crs(epsg=3857)

# Project centroids to 3857 for distance calculations
gdf_centroids_proj = gdf_centroids.to_crs(epsg=3857)

# Build KDTree on station coordinates (projected) and query nearest neighbor 
# Coordinates in meters
station_coords = np.vstack([gdf_stations.geometry.x.values, gdf_stations.geometry.y.values]).T
tract_coords = np.vstack([gdf_centroids_proj.geometry.x.values, gdf_centroids_proj.geometry.y.values]).T

tree = cKDTree(station_coords)
distances, idxs = tree.query(tract_coords, k=1)

# Assign nearest station name and distance (meters)
gdf_centroids['nearest_station'] = gdf_stations.iloc[idxs].station.values
gdf_centroids['distance_to_station_m'] = distances

# Save mapping table and (optionally) merge to your outages dataframe 
tract_station_map = gdf_centroids[['TRACT', 'tract_lat', 'tract_lon', 'nearest_station', 'distance_to_station_m']].copy()
tract_station_map['TRACT'] = tract_station_map['TRACT'].astype(str)

#Take out tract and county from block FIPS
tract_station_map['TRACT'] = tract_station_map['TRACT'].str[5:11]

# Example: merge into outage dataframe (df_out or whatever you call it)
# df_out currently has column 'TRACT' — make sure types match (strings)
#df_out['TRACT'] = df_out['TRACT'].astype(str)
#tract_station_map['TRACT'] = tract_station_map['TRACT'].astype(str)

#df_out = df_out.merge(tract_station_map, on='TRACT', how='left')

# ------------- 6) Quick checks -------------
# How many tracts failed to map?
#n_unmapped = df_out['nearest_station'].isna().sum()
#print(f"Number of outage rows without a mapped station: {n_unmapped}")

# View a sample of mappings
#print(tract_station_map.sample(5))

Hurricane Helene impacted the Southeast U.S. roughly:

UTC window to remove:

Start: 2024-09-26 00:00 UTC

End: 2024-09-30 23:59 UTC

In [16]:
combined_df['observation_datetime'] = pd.to_datetime(combined_df['observation_datetime'])

# Define UTC window of hurricane
start_helene = pd.Timestamp("2024-09-26 00:00")
end_helene   = pd.Timestamp("2024-09-30 23:59")

# Filter out the Helene rows
combined_df_filtered = combined_df[
    ~combined_df['observation_datetime'].between(start_helene, end_helene)
]

#so now join tract to combined_df with help of tract_station_map, and then find


In [17]:
# prepare daily station weather (station x date)
combined_df_filtered['observation_datetime'] = pd.to_datetime(combined_df_filtered['observation_datetime'], utc=True)
combined_df_filtered['date'] = combined_df_filtered['observation_datetime'].dt.date

weather_daily = combined_df_filtered.groupby(['station','date'], as_index=False).agg(
    mean_wind = ('windspeed10m_mph', 'mean'),
    max_gust  = ('gustspeed10m_mph', 'max'),
    total_precip = ('precip_in', 'sum'),
    mean_soilmoist = ('soilmoist20cm', 'mean')
)

# ensure TRACT and station mapping present on outages
outage_events['TRACT'] = outage_events['TRACT'].astype(str)
tract_station_map['TRACT'] = tract_station_map['TRACT'].astype(str)

if 'nearest_station' not in outage_events.columns:
    outage_events = outage_events.merge(
        tract_station_map[['TRACT','nearest_station']],
        on='TRACT', how='left'
    )

# make outage-days (one row per outage-day)
outage_events['start_date'] = pd.to_datetime(outage_events['outage_start_estimate'], utc=True).dt.date
outage_events['end_date']   = pd.to_datetime(outage_events['outage_end_estimate'], utc=True).dt.date

outage_events['outage_dates'] = outage_events.apply(
    lambda r: pd.date_range(r['start_date'], r['end_date'], freq='D').date, axis=1
)

outage_days = outage_events[['outage_identifer','TRACT','nearest_station','outage_dates']].explode('outage_dates')
outage_days = outage_days.rename(columns={'outage_dates':'date'}).dropna(subset=['nearest_station'])

# join outage-days to station daily weather
outage_weather_daily = outage_days.merge(
    weather_daily,
    left_on=['nearest_station','date'],
    right_on=['station','date'],
    how='left'
)

# aggregate to one row per TRACT
tract_weather_summary = outage_weather_daily.groupby('TRACT', as_index=False).agg(
    max_gust_during_outage_days = ('max_gust', 'max'),
    mean_wind_on_outage_days    = ('mean_wind', 'mean'),
    total_precip_on_outage_days = ('total_precip', 'sum'),
    mean_soilmoist_on_outage_days = ('mean_soilmoist', 'mean'),
    outage_weather_day_count = ('date', lambda x: x.nunique())
)

/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_16720/1258533310.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['observation_datetime'] = pd.to_datetime(combined_df_filtered['observation_datetime'], utc=True)
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_16720/1258533310.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['date'] = combined_df_filtered['observation_datetime'].dt.date


In [18]:
tract_weather = tract_weather_summary

# Save dataframes to csvs for later use

In [10]:
tract_socioeconomic.to_csv('data_processed/PCA_data/tract_socioeconomic.csv', index=False)
tract_socioeconomic_unscaled.to_csv('data_processed/PCA_data/tract_socioeconomic_unscaled.csv', index=False)
#outage_events.to_csv('data_processed/PCA_data/outage_events.csv', index=False)
#tract_outages.to_csv('data_processed/PCA_data/tract_outages.csv', index=False)
#tract_weather.to_csv('data_processed/PCA_data/tract_weather.csv', index=False)